# Inventory Health Analysis — Distressed Retail Client

**For:** Areté Capital Partners — Data & AI Team  
**Audience:** Data/AI lead + non-technical consultant  
**Window:** Jan–Dec 2024

## Approach

Three systems, no shared identifier. The task is (1) reconcile them into a single SKU-keyed view, (2) surface findings a CFO will act on, and (3) leave behind code a teammate can point at the next client.

The reusable pieces live in the `retail_reconcile/` package. Client-specific mappings (column names, sheet names, SKU rules) are isolated to `config.py` and the inventory loader. Everything else — identifier normalization, date parsing, fuzzy cross-source matching, and the insight functions — is intended to work unchanged on the next engagement.

In [2]:
import json
import pandas as pd
from retail_reconcile import (
    load_pos, load_inventory, load_ecommerce,
    stockout_risk, dead_inventory, reconciliation_gaps,
    channel_performance, data_quality_report,
)
from retail_reconcile.reconcile import reconcile_sources
from retail_reconcile.ai_insights import summarize_for_cfo, triage_ops_notes
pd.set_option('display.max_columns', 20)

## 1. Load and reconcile

Each loader produces a canonical schema so downstream code doesn't care about source format.

In [3]:
pos = load_pos('pos_transactions.csv')
inv = load_inventory('inventory_management.xlsx')
ecom = load_ecommerce('ecommerce_export.json')
art = reconcile_sources(pos, inv, ecom)
print(f'POS transactions: {len(pos):,}')
print(f'Inventory SKUs:   {len(inv):,}')
print(f'E-com orders:     {len(ecom):,}')
print(f"E-com → SKU mapped: {art['ecom_map']['matched_sku'].notna().sum()}/{len(art['ecom_map'])} "
      f'via fuzzy product-name match')

POS transactions: 502,505
Inventory SKUs:   265
E-com orders:     125,000
E-com → SKU mapped: 212/212 via fuzzy product-name match


### Reconciliation strategy

POS SKUs came in five flavours — `SKU-50128`, `SKU50128`, `50128`, `050128`, `50128C`. A single `normalize_sku` function (see `normalize.py`) produces the canonical form; ~96% of unique POS SKUs then line up directly against the inventory system.

E-commerce uses a completely separate namespace (`ECOM-XXXXXX`). The only bridge is `product_name`. We use RapidFuzz `token_sort_ratio` with an 88/100 cutoff and match 212/212 unique e-commerce products to inventory SKUs. That threshold was picked after eyeballing the close-call matches at lower scores — 85 and below produced false positives ("Classic Pen Holder" vs "Classic Pencil Holder").

## 2. Data quality report

Before any insight, be transparent about what's broken in the input.

In [4]:
dq = data_quality_report(art)
print(json.dumps(dq, indent=2))

{
  "pos": {
    "total_rows": 502505,
    "missing_sku": 1,
    "missing_store": 100580,
    "missing_customer": 125684,
    "negative_quantity_rows": 40179,
    "unparseable_date": 4,
    "orphan_skus_vs_inventory": 4
  },
  "ecommerce": {
    "unique_products": 212,
    "mapped_to_sku": 212,
    "unmapped": 0
  },
  "inventory": {
    "skus": 265,
    "rows_with_ops_notes": 86,
    "missing_reorder_level": 0
  }
}


### What these numbers mean

- **40,179 POS rows (8%) have negative quantity.** Some are legitimate returns, but the POS system doesn't flag them distinctly — the client can't tell a return from a keying error without the original transaction. We net them when computing velocity, because that matches cashflow reality, but the CFO should know the data can't separate the two.
- **125,684 POS transactions (25%) have no customer_id.** Cash walk-ins are fine; 25% is too high, which suggests POS operators are skipping the loyalty lookup. This kills repeat-customer analytics.
- **Three date formats in one column:** ISO, `dd-mm-yyyy`, and `mm/dd/yyyy`. Parsed defensively in `normalize.py`.
- **4 POS SKUs (including `SAMPLE`) never appeared in inventory.** See orphan sales below.

## 3. Stockout risk — the biggest dollar item

In [4]:
so = stockout_risk(art)
print(f'SKUs with <14 days cover: {len(so)}')
print(f'30-day lost revenue if stockouts hit: ${so["lost_revenue_30d_if_out"].sum():,.0f}')
so.head(10)

SKUs with <14 days cover: 250
30-day lost revenue if stockouts hit: $20,513,339


,sku,product_name,qty_on_hand,reorder_level,units_30d,daily_velocity,days_cover,retail_price,lost_revenue_30d_if_out
62,45776,Handmade Candle Holder,0,10,1218.0,40.600000,0.000000,145.57,177304.26
140,31598,Organic Figurine,1,19,1163.0,38.766667,0.025795,148.53,172740.39
36,91396,Vintage Vase,1,21,1112.0,37.066667,0.026978,23.85,26521.20
35,44548,Deluxe Garden Tools,1,34,1070.0,35.666667,0.028037,138.58,148280.60
233,83792,Large Basket,2,42,909.0,30.300000,0.066007,27.44,24942.96
125,35441,Classic Garden Tools,3,43,1299.0,43.300000,0.069284,46.90,60923.10
229,11747,Classic Coaster Set,4,24,1238.0,41.266667,0.096931,37.94,46969.72
197,79093,Set Of Ornament,4,40,1096.0,36.533333,0.109489,96.30,105544.80
40,82482,Large Vase,6,20,985.0,32.833333,0.182741,110.59,108931.15
91,91454,Mini Storage Box,8,25,1243.0,41.433333,0.193081,109.63,136270.09


Definition: `days_cover = qty_on_hand / (last-30-day units sold ÷ 30)`. The ten SKUs at the top are effectively stocked out already — 40+ units/day of sell-through against 0–8 on hand. Expediting those POs is worth roughly $1M in recovered 30-day revenue.

## 4. Dead inventory — unlaunched products

In [5]:
di = dead_inventory(art)
print(f'SKUs with stock but no sale in 60 days: {len(di)}')
print(f'Capital tied up: ${di["capital_tied_up"].sum():,.0f}')
di.head(15)

SKUs with stock but no sale in 60 days: 15
Capital tied up: $49,114


,sku,product_name,category,qty_on_hand,unit_cost,capital_tied_up,last_sale,days_since_sale
262,NEW90012,New Product 13 - Not Yet Active,Bedroom,156,49.03,7648.68,NaT,NaN
250,NEW90000,New Product 1 - Not Yet Active,Home Decor,166,41.57,6900.62,NaT,NaN
254,NEW90004,New Product 5 - Not Yet Active,Kitchen,175,33.82,5918.50,NaT,NaN
256,NEW90006,New Product 7 - Not Yet Active,Kids,180,25.11,4519.80,NaT,NaN
260,NEW90010,New Product 11 - Not Yet Active,Gifts,134,24.80,3323.20,NaT,NaN
252,NEW90002,New Product 3 - Not Yet Active,Kitchen,86,31.77,2732.22,NaT,NaN
257,NEW90007,New Product 8 - Not Yet Active,Kitchen,106,25.30,2681.80,NaT,NaN
263,NEW90013,New Product 14 - Not Yet Active,Bedroom,181,14.05,2543.05,NaT,NaN
261,NEW90011,New Product 12 - Not Yet Active,Bedroom,57,40.04,2282.28,NaT,NaN
259,NEW90009,New Product 10 - Not Yet Active,Office,53,39.23,2079.19,NaT,NaN


The whole list is `New Product N — Not Yet Active` — products the client paid for and stocked but never put on sale. $49K in working capital is parked in launches that never happened. Tiny dollar figure next to stockout risk, but the organizational story matters: something in the product-launch workflow is broken.

## 5. Reconciliation gaps — where the systems disagree

In [6]:
rg = reconciliation_gaps(art)
print(f'Gap rows: {len(rg)}; total $ impact: ${rg["dollar_impact"].sum():,.0f}')
rg.head(12)

Gap rows: 43; total $ impact: $23,300


,sku,product_name,system_qty,ops_physical_count,delta_units,dollar_impact,gap_type,evidence
39,SAMPLE,Sample Product - Training,NaN,NaN,NaN,9999.99,pos_orphan,"1 POS transactions, $10000 revenue, no invento..."
9,41747,Premium Cutlery,108.0,89.0,-19.0,1322.59,ops_override,Physical count: 89 (system wrong)
17,31286,Premium Plate Set,41.0,23.0,-18.0,945.00,ops_override,Physical count: 23 (system wrong)
7,95812,Modern Lamp,134.0,150.0,16.0,737.12,ops_override,Physical count: 150 (system wrong)
12,49689,Deluxe Diffuser,182.0,202.0,20.0,720.60,ops_override,Physical count: 202 (system wrong)
34,79093,Set Of Ornament,4.0,17.0,13.0,650.00,ops_override,Physical count: 17 (system wrong)
2,34053,Handmade Desk Organizer,142.0,131.0,-11.0,617.76,ops_override,Physical count: 131 (system wrong)
10,62995,Budget Candle,34.0,19.0,-15.0,574.20,ops_override,Physical count: 19 (system wrong)
1,20951,Mini Wall Art,17.0,33.0,16.0,538.72,ops_override,Physical count: 33 (system wrong)
18,52019,Set Of Garden Tools,139.0,145.0,6.0,472.32,ops_override,Physical count: 145 (system wrong)


Two kinds of gaps:

1. **Ops overrides in the Notes column.** The ops team has been quietly running a parallel accounting system in spreadsheet prose because they don't trust the inventory system. 39 SKUs have physical counts that disagree with system qty, $13K of inventory variance.
2. **POS orphans — sales without inventory records.** Including a SKU literally named `SAMPLE` that moved $10K of real revenue. Either the product exists and someone never set it up in inventory, or it's a register-test SKU that operators used for live transactions. Both are bad.

## 6. Channel performance

In [7]:
cp = channel_performance(art)
cp

,category,instore,online,total,online_share
9,Seasonal,71897805.15,3445629.99,75343435.14,0.046
4,Home Decor,70679500.22,2763894.29,73443394.51,0.038
6,Kitchen,66127329.77,1645986.30,67773316.07,0.024
2,Garden,63960149.59,2634762.91,66594912.50,0.040
5,Kids,61491458.36,2740764.23,64232222.59,0.043
0,Bathroom,52057747.92,3245434.03,55303181.95,0.059
1,Bedroom,50580404.61,2018175.08,52598579.69,0.038
3,Gifts,46143550.16,2010588.87,48154139.03,0.042
8,Outdoor,45421318.45,1938640.12,47359958.57,0.041
7,Office,42294323.05,1369626.87,43663949.92,0.031


In [8]:
total = cp['total'].sum()
online = cp['online'].sum()
print(f'Total 2024 revenue: ${total/1e6:.0f}M')
print(f'Online share: {online/total:.1%} (${online/1e6:.1f}M)')
print('Online share is well below US retail average (~15%). ',
      'Either the e-commerce export is incomplete, or the online channel is materially under-invested.')

Total 2024 revenue: $594M
Online share: 4.0% ($23.8M)
Online share is well below US retail average (~15%).  Either the e-commerce export is incomplete, or the online channel is materially under-invested.


## 7. AI-assisted insight generation

Two places the LLM earns its keep:

1. **Triaging the ops Notes column.** A regex handles the ~80% `Physical count: N (system wrong)` pattern. For the long tail (`Display only - do not sell`, `Reserved for customer order`, `Awaiting vendor credit`, `Adj: -14 per Lisa 1/14`), regex is fragile; the LLM classifies and extracts recommended actions into a Pydantic-typed record.
2. **CFO narrative.** Turns the numeric findings dict into a one-paragraph summary in plain English.

Both outputs are constrained by Pydantic schemas (`ai_insights.py`) so we can validate before trusting. Offline fallbacks exist so the notebook runs without an API key.

In [9]:
notes_df = art['inventory'][['sku', 'notes']].dropna(subset=['notes'])
print(f'Notes to classify: {len(notes_df)}')
triage = triage_ops_notes(notes_df, limit=20)
pd.DataFrame([t.model_dump() for t in triage]).head(20)

Notes to classify: 86


,sku,raw_note,issue_type,severity,recommended_action,confidence
0,46271,Physical count: 33 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55
1,20951,Physical count: 33 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55
2,34053,Physical count: 131 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55
3,90336,Physical count: 33 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55
4,38217,Seasonal item - store after Dec,other,medium,Reconcile system quantity with physical count.,0.55
5,76021,Reserved for customer order,other,medium,Reconcile system quantity with physical count.,0.55
6,19333,Display only - do not sell,other,medium,Reconcile system quantity with physical count.,0.55
7,90479,Physical count: 122 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55
8,16295,Physical count: 77 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55
9,95676,Physical count: 187 (system wrong),physical_count_override,high,Reconcile system quantity with physical count.,0.55


In [10]:
findings = {
    'at_risk_skus': len(so),
    'lost_revenue_at_risk': float(so['lost_revenue_30d_if_out'].sum()),
    'dead_skus': len(di),
    'dead_capital': float(di['capital_tied_up'].sum()),
    'ops_override_count': int((rg['gap_type']=='ops_override').sum()),
    'ops_override_dollars': float(rg[rg['gap_type']=='ops_override']['dollar_impact'].sum()),
    'pos_negative_qty_rows': dq['pos']['negative_quantity_rows'],
    'pos_missing_customer': dq['pos']['missing_customer'],
    'pct_missing_customer': dq['pos']['missing_customer'] / dq['pos']['total_rows'],
    'ecom_unmapped': dq['ecommerce']['unmapped'],
    'total_revenue': float(cp['total'].sum()),
    'online_share': float(cp['online'].sum() / cp['total'].sum()),
}
summary = summarize_for_cfo(findings)
print(summary.model_dump_json(indent=2))

{
  "headline": "$20.5M of revenue is at stockout risk across 250 SKUs, while $49,114 sits in unlaunched or non-moving inventory.",
  "top_risks": [
    "250 SKUs have under 14 days of cover — potential $20.5M lost revenue.",
    "15 SKUs have not sold in 60 days, tying up $49,114 of working capital.",
    "System quantity disagrees with ops physical counts on 39 SKUs worth $13,300."
  ],
  "data_quality_callouts": [
    "40,179 POS rows have negative quantities — returns are not flagged distinctly from data-entry errors.",
    "125,684 POS transactions (25%) have no customer ID — limits loyalty/repeat analysis.",
    "E-commerce uses a separate product-ID namespace; 0 products still cannot be mapped to inventory automatically."
  ],
  "recommended_next_steps": [
    "Expedite purchase orders for the top 10 at-risk SKUs this week.",
    "Return or liquidate the 15 dead SKUs to recover working capital.",
    "Enforce a single SKU format at POS entry; add required-field validation for cu

### What the AI did well vs. what I didn't trust

**Did well:**
- Classifying free-text ops notes into a typed taxonomy was worth the tokens. `Display only - do not sell` and `Reserved for customer order` are semantically critical — these SKUs shouldn't be counted as sellable inventory — and regex would miss them without an enumerated list.
- Narrative generation with a constrained Pydantic schema + dollar figures passed in as source-of-truth data produced CFO-ready copy.

**Did not trust:**
- I never let the LLM compute a number. All $ figures and counts are pandas outputs; the LLM only shapes them into prose.
- I never let the LLM pick which SKU "matters." Ranking is a deterministic sort by dollar impact.
- For cross-source fuzzy matching I used RapidFuzz, not an embedding model. Deterministic, tunable threshold, and good enough on 212 product names — an LLM adds cost without clear accuracy gain at this N.

## 8. Design — what's reusable vs. client-specific

| Module | Reusable? | Why |
|---|---|---|
| `normalize.py` | Yes | SKU/date/name cleaning is industry-generic. |
| `reconcile.py` | Yes | Works against any two-namespace product system. |
| `insights.py` | Yes | Metrics are common retail analytics; thresholds are config. |
| `ai_insights.py` | Yes | Pydantic schemas + fallbacks, no client facts hardcoded. |
| `config.py` | **Client-specific** | SKU prefixes, thresholds, source priority. |
| `loaders.load_inventory` | **Partially** | Column renames are per-vendor. Factor out into YAML for production. |

With more time I'd (1) pull per-client column renames into a YAML config so `loaders.py` becomes a dispatcher, (2) write unit tests for `normalize_sku` — it has enough branches to regress, and (3) add a reorder-point calculator keyed to service-level target rather than the client's flat reorder levels.